# Logistic Regression

**Companion lesson:** https://ml-viz.vercel.app/courses/linear-regression/02-logistic-regression

This notebook implements logistic regression *from scratch* with gradient descent and
mirrors the math derived in the lesson: the sigmoid + log-odds link, binary cross-entropy
from MLE, and the clean gradient collapse `(1/n) X^T (sigma(Xw) - y)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark matplotlib style to match the site
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## 1. The sigmoid function and the log-odds link

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \log\frac{\sigma(z)}{1-\sigma(z)} = z$$

The sigmoid squashes any real score into $(0,1)$, and its log-odds (logit) is exactly the
linear score $z$. We verify the log-odds identity numerically below.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
sig = sigmoid(z)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sig, color='#818cf8', linewidth=2.5)
ax.axhline(0.5, color='#2e3347', linestyle='--')
ax.axvline(0, color='#2e3347', linestyle='--')
ax.set_title('Sigmoid Function', color='white')
ax.set_xlabel('z')
ax.set_ylabel('sigma(z)')
plt.tight_layout()
plt.show()

# log-odds(sigma(z)) should equal z
logit = np.log(sig / (1 - sig))
print('max |logit(sigma(z)) - z| = {:.2e}'.format(np.max(np.abs(logit - z))))

## 2. Cross-entropy penalizes confident mistakes

Binary cross-entropy is the negative log-likelihood of a Bernoulli model:

$$\mathcal{L} = -\frac{1}{n}\sum_i \big[ y_i \log \hat y_i + (1-y_i)\log(1-\hat y_i) \big]$$

For a positive example the loss is $-\log(\hat y)$, which explodes as the model becomes
confidently wrong.

In [ ]:
def cross_entropy(y_true, y_pred):
    eps = 1e-15  # avoid log(0)
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

p = np.linspace(1e-3, 1 - 1e-3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, -np.log(p), color='#818cf8', label='y=1: -log(p)')
ax.plot(p, -np.log(1 - p), color='#f43f5e', label='y=0: -log(1-p)')
ax.set_xlabel('predicted P(y=1)')
ax.set_ylabel('loss')
ax.set_ylim(0, 5)
ax.set_title('Cross-entropy loss', color='white')
ax.legend()
plt.tight_layout()
plt.show()

print('loss when predicting 0.01 for a positive: {:.2f}'.format(-np.log(0.01)))

## 3. A tiny worked gradient step (matches the lesson by hand)

With

$$X = \begin{bmatrix} 1 & 2 \\ 1 & -1 \end{bmatrix},\; y = [1, 0],\; w = [0,0],\; \eta = 0.5$$

every score is 0, so $\hat y = 0.5$. The gradient is $\tfrac1n X^\top(\hat y - y) = [0, -0.75]$,
giving the update $w \leftarrow [0, 0.375]$. We confirm it.

In [ ]:
X_demo = np.array([[1.0, 2.0],
                   [1.0, -1.0]])
y_demo = np.array([1.0, 0.0])
w_demo = np.zeros(2)
eta = 0.5
n_demo = X_demo.shape[0]

y_hat = sigmoid(X_demo @ w_demo)
grad = (1 / n_demo) * X_demo.T @ (y_hat - y_demo)
w_new = w_demo - eta * grad

print('y_hat       =', y_hat)
print('gradient    =', grad)          # expect [0, -0.75]
print('updated w   =', w_new)         # expect [0, 0.375]

## 4. Logistic regression from scratch via gradient descent

We generate two Gaussian blobs, prepend a bias column, and run the update
$w \leftarrow w - \eta \cdot \tfrac1n X^\top(\sigma(Xw) - y)$ while recording the loss.

In [ ]:
np.random.seed(42)
n = 200
X_pos = np.random.randn(n // 2, 2) + [1.5, 1.0]
X_neg = np.random.randn(n // 2, 2) + [-1.5, -1.0]
X_raw = np.vstack([X_pos, X_neg])
y = np.array([1.0] * (n // 2) + [0.0] * (n // 2))

# design matrix with bias column of ones
X_b = np.c_[np.ones(n), X_raw]
w = np.zeros(X_b.shape[1])
lr = 0.2
epochs = 500
losses = []

for epoch in range(epochs):
    y_pred = sigmoid(X_b @ w)
    grad = (1 / n) * X_b.T @ (y_pred - y)   # the clean collapse
    w -= lr * grad
    losses.append(cross_entropy(y, y_pred))
    if epoch % 100 == 0:
        acc = np.mean((y_pred >= 0.5) == y)
        print('epoch {:3d}: loss={:.4f}  acc={:.3f}'.format(epoch, losses[-1], acc))

final_acc = np.mean((sigmoid(X_b @ w) >= 0.5) == y)
print('final weights [bias, w1, w2] = {}'.format(w.round(3)))
print('final accuracy = {:.3f}'.format(final_acc))

## 5. Loss curve

Cross-entropy is convex in $w$, so the loss decreases smoothly to a single minimum.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='#14b8a6', linewidth=2)
ax.set_xlabel('epoch')
ax.set_ylabel('cross-entropy loss')
ax.set_title('Training loss', color='white')
plt.tight_layout()
plt.show()

## 6. Decision boundary

The boundary is the line $w^\top x = 0$. We shade $P(y=1)$ and draw the 0.5 contour.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pos[:, 0], X_pos[:, 1], c='#818cf8', s=30, alpha=0.7, label='Class 1')
ax.scatter(X_neg[:, 0], X_neg[:, 1], c='#f43f5e', s=30, alpha=0.7, label='Class 0')

x0, x1 = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
grid = np.c_[np.ones(x0.size), x0.ravel(), x1.ravel()]
probs = sigmoid(grid @ w).reshape(x0.shape)

ax.contourf(x0, x1, probs, levels=20, alpha=0.18, cmap='RdYlBu')
ax.contour(x0, x1, probs, levels=[0.5], colors=['#14b8a6'], linewidths=2)
ax.set_title('Logistic Regression Decision Boundary', color='white')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.legend()
plt.tight_layout()
plt.show()

## Key takeaways

- Logistic regression passes a linear score through the **sigmoid** to get $P(y=1)\in(0,1)$.
- It is trained with **cross-entropy** loss, the negative log-likelihood of a Bernoulli model.
- The gradient collapses to $\tfrac1n X^\top(\hat y - y)$ — the same form as linear regression.
- The decision boundary is **linear**: the hyperplane $w^\top x = 0$.
- **Softmax** generalizes the sigmoid to multi-class classification.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Binary cross-entropy

The loss that trains logistic regression averages the surprise of each prediction:

$$L = -\frac{1}{n} \sum_i \Big[\, y_i \log p_i + (1 - y_i) \log(1 - p_i) \,\Big]$$

Implement it. The checks pin the landmarks from section 2: perfect predictions cost ~0, coin-flip predictions cost exactly $\log 2$, and a *confident* mistake costs an order of magnitude more than a mild one.

In [ ]:
def bce(y, p):
    """Binary cross-entropy between labels y (0/1) and predicted probabilities p."""
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    # TODO(you): -mean of y*log(p) + (1-y)*log(1-p)
    return ...

In [ ]:
# Checks — run me
assert bce([1, 0, 1], [0.999999, 0.000001, 0.999999]) < 1e-5, "near-perfect predictions -> near-zero loss"
assert abs(bce([1, 0], [0.5, 0.5]) - np.log(2)) < 1e-12, "coin-flip predictions cost log 2 ≈ 0.693"

confident_wrong = bce([1.0], [0.01])
mild_wrong = bce([1.0], [0.4])
assert confident_wrong > 4 and mild_wrong < 1, "confident mistakes are punished much harder"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bce(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
```

</details>

### Exercise 2 — The logistic-regression gradient

Cross-entropy plus sigmoid collapses into the same elegant form as linear regression — *prediction minus target*:

$$\nabla_{\mathbf{w}} L = \frac{1}{n} X^\top (\mathbf{p} - \mathbf{y}), \qquad \mathbf{p} = \sigma(X\mathbf{w})$$

Implement it. The checks compare every component against finite differences of the loss, and confirm a small step against your gradient actually lowers it.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logistic_gradient(X, y, w):
    """Gradient of mean binary cross-entropy wrt w."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): predicted probabilities p = sigmoid(X @ w)
    p = ...

    # TODO(you): X^T (p - y) / n
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(1)
Xl = np.column_stack([np.ones(40), rng.standard_normal((40, 2))])
yl = (Xl[:, 1] + Xl[:, 2] > 0).astype(float)
w0 = np.array([0.1, -0.2, 0.3])

def nll(w):
    p = sigmoid(Xl @ w)
    return -np.mean(yl * np.log(p) + (1 - yl) * np.log(1 - p))

g = logistic_gradient(Xl, yl, w0)
h = 1e-6
for i in range(3):
    e = np.zeros(3); e[i] = h
    num = (nll(w0 + e) - nll(w0 - e)) / (2 * h)
    assert abs(g[i] - num) < 1e-6, f"component {i} must match the numerical gradient"

assert nll(w0 - 0.5 * g) < nll(w0), "a small step against the gradient lowers the loss"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def logistic_gradient(X, y, w):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    p = sigmoid(X @ w)
    return X.T @ (p - y) / len(y)
```

</details>